In [1]:
import pandas as pd
import os
import folium
import sqlite3

In [2]:
with sqlite3.connect("../data/processed/flight_data.db") as conn:
    # 2. Run a query and load directly into a DataFrame
    query = 'SELECT * from us_routes'
    routes_df = pd.read_sql_query(query, conn)
routes_df.head()

DatabaseError: Execution failed on sql 'SELECT * from us_routes': no such table: us_routes

In [ ]:
with sqlite3.connect("../data/processed/flight_data.db") as conn:
    # 2. Run a query and load directly into a DataFrame
    query = 'SELECT * from large_airports'
    airports_df = pd.read_sql_query(query, conn)
airports_df.head()

In [ ]:
map = folium.Map(location=[44.9672, -103.7716], zoom_start=3)

In [3]:
# 2. Create feature groups
airports = folium.FeatureGroup(name="Airports")

# 4. Add groups to map
map.add_child(airports)

def apply_to_each_row(row):
    iata = row["iata_code"]
    lat = row["latitude_deg"]
    long = row["longitude_deg"]
    name = row["name"]
    if iata is not None and lat is not None and long is not None and name is not None:
        folium.Marker(
            location=[float(lat), float(long)],
            popup=iata,
            tooltip=name,
            icon=folium.Icon(color="blue", icon="info-sign")
).add_to(airports)

airports_df.apply(apply_to_each_row, axis=1)

from pyproj import Geod

routes = folium.FeatureGroup(name="Routes")

# 4. Add groups to map
map.add_child(routes)

# Define endpoints: [latitude, longitude]
boston = [42.3581, -71.0636]
sf = [37.7833, -122.4167]

# Initialize WGS84 geodesic model
geod = Geod(ellps="WGS84")

# pyproj uses (lon, lat) order for npts
# npts calculates N intermediate points between start and end
num_intermediate_points = 13
points = geod.npts(
    lon1=boston[1],
    lat1=boston[0],
    lon2=sf[1],
    lat2=sf[0],
    npts=num_intermediate_points,
)

# Combine start, intermediate points, and end into [lat, lon] format for Folium
coordinates = [boston] + [[lat, lon] for lon, lat in points] + [sf]

folium.PolyLine(
    locations=coordinates,
    color="#FF0000",
    weight=5,
    tooltip="From Boston to San Francisco",
).add_to(routes)
folium.LayerControl().add_to(map)
map

AttributeError: type object 'map' has no attribute 'add_child'